# **1. Tiền Xử Lý Dữ Liệu**

**Đồ án 2 – Dự báo ngắn hạn:** dự báo **số khách** (`Customers`) và **doanh thu** (`Sales`) theo ngày cho từng cửa hàng, cho **ngày mai** (h = 1) và **tuần tới** (h = 7). Kết quả dùng để xếp lịch nhân viên và chuẩn bị lượng hàng.

Notebook này đọc dữ liệu gốc, kiểm tra, tạo **lịch đầy đủ** cho từng cửa hàng và lưu lại cho các notebook sau.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

import features as F            # code tạo đặc trưng, dùng chung với web demo (app.py)

RAW_DIR = Path('..')            # train.csv, store.csv, test.csv để ở thư mục gốc, dùng chung với đồ án 1
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

Đọc thông tin của chuỗi cửa hàng.

In [2]:
info_data = pd.read_csv(RAW_DIR / 'store.csv')
print(info_data.shape)
info_data.head()

(1115, 10)


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


Đọc dữ liệu bán hàng theo ngày. Khác với đồ án 1, cột `Customers` được **giữ lại** vì đây là một trong hai đại lượng cần dự báo.

In [3]:
train_data = pd.read_csv(RAW_DIR / 'train.csv', parse_dates=['Date'], dtype={'StateHoliday': str})
print(train_data.info())
print('Từ', train_data['Date'].min().date(), 'đến', train_data['Date'].max().date(),
      '|', train_data['Store'].nunique(), 'cửa hàng')

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[us]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  str           
 8   SchoolHoliday  1017209 non-null  int64         
dtypes: datetime64[us](1), int64(7), str(1)
memory usage: 70.8 MB
None
Từ 2013-01-01 đến 2015-07-31 | 1115 cửa hàng


In [4]:
open_days = train_data[train_data['Open'] == 1]
open_days[['Sales', 'Customers']].describe().round(1)

,Sales,Customers
count,844392.0,844392.0
mean,6955.5,762.7
std,3104.2,401.2
min,0.0,0.0
25%,4859.0,519.0
50%,6369.0,676.0
75%,8360.0,893.0
max,41551.0,7388.0


Kiểm tra dữ liệu bị thiếu và trùng lặp.

In [5]:
my_data = pd.merge(info_data, train_data, on='Store', how='left')
print(my_data.isnull().sum()[lambda s: s > 0])
del my_data                       # bảng ghép rất lớn, kiểm tra xong thì giải phóng bộ nhớ

# store.csv có đúng 1 dòng mỗi cửa hàng, nên kiểm tra trùng lặp trên train_data là đủ (nhẹ hơn nhiều)
print('Số dòng trùng lặp hoàn toàn:', train_data.duplicated().sum())
print('Số cặp (Store, Date) bị lặp:', train_data.duplicated(subset=['Store', 'Date']).sum())

CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64


Số dòng trùng lặp hoàn toàn: 0
Số cặp (Store, Date) bị lặp: 0


**Kiểm tra số ngày của từng cửa hàng.** Đồ án này dùng độ trễ ngắn (`lag_1`, `lag_7`…), và độ trễ phải đếm theo **ngày lịch**. Nếu một cửa hàng bị thiếu ngày thì `shift(7)` sẽ không còn là "7 ngày trước". Vì vậy cần biết cửa hàng nào bị thiếu ngày.

In [6]:
n_days = train_data.groupby('Store').size()
print(n_days.value_counts())

store_gap = n_days.idxmin()
dates = train_data.loc[train_data['Store'] == store_gap, 'Date'].sort_values()
jump = dates.diff().dt.days
print(f'Ví dụ cửa hàng {store_gap}: nhảy từ {dates[jump.idxmax()] - pd.Timedelta(days=int(jump.max()))} '
      f'sang {dates[jump.idxmax()]} ({int(jump.max()) - 1} ngày bị thiếu)')

942    934
758    180
941      1
Name: count, dtype: int64
Ví dụ cửa hàng 13: nhảy từ 2014-06-30 00:00:00 sang 2015-01-01 00:00:00 (184 ngày bị thiếu)


**Nhận xét:** 934 cửa hàng có đủ 942 ngày, nhưng 180 cửa hàng chỉ có 758 ngày vì thiếu trọn 184 ngày từ 01/07 đến 31/12/2014 (ví dụ cửa hàng 13), và 1 cửa hàng thiếu 1 ngày. Nếu `shift(7)` trực tiếp trên dữ liệu này thì với 180 cửa hàng đó, "lag_7" của những ngày đầu tháng 1/2015 sẽ là số liệu của cuối tháng 6/2014, sai hoàn toàn. Vì vậy phải chèn đủ ngày trước khi tạo độ trễ.

Kiểm tra các ngày **mở cửa nhưng không bán được hàng** (dữ liệu bất thường).

In [7]:
odd = train_data[(train_data['Open'] == 1) & (train_data['Sales'] == 0)]
print('Mở cửa nhưng Sales = 0:', len(odd))
print('Mở cửa nhưng Customers = 0:', ((train_data['Open'] == 1) & (train_data['Customers'] == 0)).sum())
print('Tỷ lệ ngày đóng cửa:', round((train_data['Open'] == 0).mean() * 100, 2), '%')

Mở cửa nhưng Sales = 0: 54
Mở cửa nhưng Customers = 0: 52
Tỷ lệ ngày đóng cửa: 16.99 %


**Tạo lịch đầy đủ.** Hàm `F.make_full_calendar` (trong `features.py`) chèn đủ **mọi ngày** cho từng cửa hàng:
- Ngày được chèn có `InCalendarGap = 1`, `Sales` và `Customers` = NaN, nên không bao giờ được dùng để huấn luyện.
- Thông tin cố định của cửa hàng lấy lại từ `store.csv`; lịch khuyến mãi lấy theo các cửa hàng khác cùng ngày.

Ghép thêm các ngày của `test.csv` (01/08–17/09/2015), với `Sales`, `Customers` = NaN. Web demo cần các dòng này để biết lịch khuyến mãi và ngày lễ của **những ngày tương lai**.

In [8]:
raw = F.load_raw(RAW_DIR, include_test=True)
full = F.make_full_calendar(raw)
print('Trước:', raw.shape, '| Sau khi tạo lịch đầy đủ:', full.shape)
print('Số dòng được chèn thêm:', int(full['InCalendarGap'].sum()))
full.loc[(full['Store'] == store_gap) & (full['Date'] == '2014-09-01'),
         ['Store', 'Date', 'Open', 'Promo', 'Sales', 'Customers', 'InCalendarGap']]

Trước: (1058297, 18) | Sau khi tạo lịch đầy đủ: (1103850, 19)
Số dòng được chèn thêm: 45553


,Store,Date,Open,Promo,Sales,Customers,InCalendarGap
12488,13,2014-09-01,0,1,NaN,NaN,1


Chọn mốc chia dữ liệu theo thời gian. Giữ **đúng mốc của đồ án 1** để so sánh được hai đồ án.

In [9]:
TRAIN_END = pd.Timestamp('2015-04-24')     # huấn luyện: 01/01/2013 – 24/04/2015
VALID_END = pd.Timestamp('2015-06-12')     # kiểm định: 25/04 – 12/06/2015 (7 tuần)
LAST_DATE = train_data['Date'].max()       # kiểm tra: 13/06 – 31/07/2015 (7 tuần)

usable = full[F.is_trainable(full)]
print('Số dòng dùng được (mở cửa, có doanh thu):', len(usable))
print('  huấn luyện:', (usable['Date'] <= TRAIN_END).sum())
print('  kiểm định :', usable['Date'].between(TRAIN_END + pd.Timedelta(days=1), VALID_END).sum())
print('  kiểm tra  :', (usable['Date'] > VALID_END).sum())

Số dòng dùng được (mở cửa, có doanh thu): 844338


  huấn luyện: 754242
  kiểm định : 43098
  kiểm tra  : 46998


Lưu dữ liệu để tái sử dụng.

In [10]:
full.to_parquet(PROCESSED_DIR / 'rossmann_full.parquet', index=False)
print('Đã lưu', PROCESSED_DIR / 'rossmann_full.parquet')

config = {'TRAIN_END': TRAIN_END, 'VALID_END': VALID_END, 'LAST_DATE': LAST_DATE}
with open(PROCESSED_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump({k: v.strftime('%Y-%m-%d') for k, v in config.items()}, f, indent=2)
print('Đã lưu', PROCESSED_DIR / 'config.json')

Đã lưu data\processed\rossmann_full.parquet
Đã lưu data\processed\config.json


**Nhận xét:** dữ liệu không trùng lặp; chỉ có 54 ngày mở cửa mà không bán được hàng, các ngày này bị loại khi huấn luyện. Lịch đầy đủ có 1.103.850 dòng (1.115 cửa hàng × 990 ngày, đến 17/09/2015), trong đó 45.553 dòng được chèn thêm: khoảng trống 6 tháng của 180 cửa hàng và những ngày tương lai của 259 cửa hàng không có trong `test.csv`. Có 844.338 dòng dùng được: 754.242 để huấn luyện, 43.098 để kiểm định và 46.998 để kiểm tra. Trung bình mỗi ngày mở cửa, một cửa hàng có 763 khách và doanh thu 6.956.